# MLP — Multi-Layer Perceptron (Turkey, Pipeline B)

**Model**: `sklearn.neural_network.MLPRegressor` / **Target**: `ngdprsaxdctrq`  
**Variables**: Cat3 (22 vars + 3 COVID = 25 total)  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: YES (StandardScaler on train fold) / **HP tuning**: YES — grid on hidden_layer_sizes × alpha / **n_lags**: 4  
**10-model averaging** / **COVID passed through unscaled**


In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
    get_features, split_for_scaler, load_data,
    PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42; np.random.seed(SEED)
TRAIN_START = '1995-01-01'; TRAIN_END = '2011-12-31'; VAL_END = '2017-12-31'
TEST_START  = '2018-01-01'; TEST_END  = '2025-12-31'
N_LAGS = 4; N_MODELS = 20
VINTAGES = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1, 'post2': 2}

print('MLP (Turkey) — Cat3, n_lags={}, scaling=YES, 20-model median ensemble'.format(N_LAGS))


MLP (Turkey) — Cat3, n_lags=4, scaling=YES, 10-model avg


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
scaled_cols, unscaled_cols = split_for_scaler(features)

# Phase A: HP tuning on train+val (1995-2017)
tune    = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
tune_f  = mean_fill_dataset(tune, tune)
tune_fl = flatten_data(tune_f, TARGET, N_LAGS)
tune_fl = tune_fl.loc[tune_fl.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)

feature_cols = [
    c for c in tune_fl.columns
    if c != 'date' and c != TARGET
    and any(c == f or c.startswith(f + '_') for f in features)
]
sc_c = [c for c in feature_cols if any(c == s or c.startswith(s + '_') for s in scaled_cols)]
us_c = [c for c in feature_cols if any(c == u or c.startswith(u + '_') for u in unscaled_cols)]

tscv = TimeSeriesSplit(n_splits=5)
best_score = np.inf; best_alpha = 0.1; best_hls = (32,)

for hls in [(16,), (32,), (32, 16)]:
    for alpha in [1e-2, 1e-1, 1.0]:
        scores = []
        for tr, va in tscv.split(tune_fl):
            sc = RobustScaler(quantile_range=(10, 90))
            X_tr_s = sc.fit_transform(tune_fl[sc_c].values[tr])
            X_tr   = np.hstack([X_tr_s, tune_fl[us_c].values[tr]]) if us_c else X_tr_s
            X_va_s = sc.transform(tune_fl[sc_c].values[va])
            X_va   = np.hstack([X_va_s, tune_fl[us_c].values[va]]) if us_c else X_va_s
            m = MLPRegressor(hidden_layer_sizes=hls, alpha=alpha, random_state=SEED, max_iter=1500,
                             early_stopping=True, validation_fraction=0.2, n_iter_no_change=30,
                             learning_rate_init=0.0005, learning_rate='adaptive')
            m.fit(X_tr, tune_fl[TARGET].values[tr])
            p = m.predict(X_va)
            scores.append(np.mean((p - tune_fl[TARGET].values[va])**2))
        if np.mean(scores) < best_score:
            best_score = np.mean(scores)
            best_alpha = alpha
            best_hls   = hls

print('Best MLP params: alpha={}, hidden_layer_sizes={}'.format(best_alpha, best_hls))


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


...

C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


Best MLP params: alpha=1.0, hidden_layer_sizes=(32, 16)


In [3]:
# Phase B: Rolling test loop with CV-selected HPs and median ensemble
test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3,6,9,12])
]['date'].tolist()

predictions = {v: [] for v in VINTAGES}; actuals_list = []; failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date'] == pd_q][TARGET].values[0])

    for vn, offset in VINTAGES.items():
        vintage_date = pd_q + pd.DateOffset(months=offset)
        train_fl, tr_fl, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, vintage_date, N_LAGS
        )

        if len(train_fl) < 10 or len(tr_fl) != 1:
            predictions[vn].append(np.nan); continue

        try:
            sc      = RobustScaler(quantile_range=(10, 90))
            X_tr_s  = sc.fit_transform(train_fl[sc_c].values)
            X_tr    = np.hstack([X_tr_s, train_fl[us_c].values]) if us_c else X_tr_s
            models  = []
            for k in range(N_MODELS):
                m = MLPRegressor(
                    hidden_layer_sizes=best_hls, activation='relu', solver='adam',
                    alpha=best_alpha, learning_rate_init=0.0005, learning_rate='adaptive',
                    early_stopping=True, validation_fraction=0.2, n_iter_no_change=30,
                    random_state=SEED+k, max_iter=1500)
                m.fit(X_tr, train_fl[TARGET].values)
                models.append(m)

            X_te_s = sc.transform(tr_fl[sc_c].values)
            X_te   = np.hstack([X_te_s, tr_fl[us_c].values]) if us_c else X_te_s
            preds  = [m.predict(X_te)[0] for m in models]
            predictions[vn].append(np.nanmedian(preds))
        except Exception:
            predictions[vn].append(np.nan); failed += 1

    if (i+1) % 8 == 0:
        print('  {} / {}'.format(i+1, len(test_quarters)))

print('Done. {} failures.'.format(failed))


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


...

C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


C:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1500) reached and the optimization hasn't converged yet.
  warnings.warn(


  32 / 32
Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({'date': test_quarters, 'actual': actuals_list, 'prediction': predictions[vn]})
    path = os.path.join(PREDICTIONS_DIR, 'mlp_{}.csv'.format(vn))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df), df['prediction'].min(), df['prediction'].max()))


Saved mlp_m1.csv (32 rows) pred=[-0.0127,0.0345]
Saved mlp_m2.csv (32 rows) pred=[-0.0209,0.0541]
Saved mlp_m3.csv (32 rows) pred=[-0.0352,0.0531]
Saved mlp_post1.csv (32 rows) pred=[-0.0349,0.0474]
Saved mlp_post2.csv (32 rows) pred=[-0.0352,0.0543]


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print('MLP (Turkey) — Evaluation by Panel & Vintage')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(vn, rmse(a[m],pp[m]), mae(a[m],pp[m]), m.sum()))


MLP (Turkey) — Evaluation by Panel & Vintage
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.008398  MAE=0.006969  N=8
  m2  RMSFE=0.006765  MAE=0.005512  N=8
  m3  RMSFE=0.008527  MAE=0.007786  N=8
  post1  RMSFE=0.009696  MAE=0.008659  N=8
  post2  RMSFE=0.009400  MAE=0.008422  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.072222  MAE=0.045473  N=7
  m2  RMSFE=0.070210  MAE=0.045302  N=7
  m3  RMSFE=0.057218  MAE=0.039666  N=7
  post1  RMSFE=0.056421  MAE=0.040639  N=7
  post2  RMSFE=0.055387  MAE=0.039486  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.016587  MAE=0.011663  N=16
  m2  RMSFE=0.015655  MAE=0.011002  N=16
  m3  RMSFE=0.016092  MAE=0.012335  N=16
  post1  RMSFE=0.017029  MAE=0.011993  N=16
  post2  RMSFE=0.017578  MAE=0.012625  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.036029  MAE=0.017764  N=32
  m2  RMSFE=0.035085  MAE=0.017552  N=32
  m3  RMSFE=0.029725  MAE=0.017576  N=32
  post1  RMSFE=0.029722  MAE=0.017813  N=32
  post2  RMSFE=0.029691  MAE=0.018085  N=